<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 6 章：使用强化学习训练推理模型

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F01_raschka.webp" width=600>

&nbsp;
## 6.1 面向 LLM 的强化学习简介

- 推理时扩展通过为每个生成答案投入更多计算来提升推理能力
- 训练时扩展通过在训练过程中投入额外计算来提升推理能力，这正是本章的重点


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F02_raschka.webp" width=600>

- 推理时扩展与训练时扩展也可以（并且应该）结合使用，例如在基于强化学习的推理训练之后应用推理时技术
- 实际上，面向 LLM 的强化学习通常作为后训练阶段，应用在预训练模型之上或指令微调之后


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F03_raschka.webp" width=600>

- 预训练通过下一词元预测构建通用知识，而强化学习通过优化答案正确性、偏好等序列级目标来改进模型行为
- 面向 LLM 的强化学习包括推理训练和偏好调优；但正如 DeepSeek-R1 所示，面向推理的强化学习也可以直接应用于预训练基础模型
- 直接在基础模型上训练推理能力会得到相对较弱但仍具备能力的模型，同时也提供了一个更简单的环境，便于理解推理阶段带来的作用


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F04_raschka.webp" width=600>

&nbsp;
### 6.1.1 使用人类反馈的原始强化学习流程（RLHF）

- RLHF 在 2022 年的 InstructGPT 工作中提出，使用人类偏好标签训练 LLM（这是将 GPT-3 转变为最初 ChatGPT 的关键步骤）
- 预训练和监督微调优化的是下一词元预测，而 RLHF 根据人类对模型回复的偏好标签来优化模型


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F05_raschka.webp" width=600>

&nbsp;
### 6.1.2 从人类反馈到可验证奖励（RLVR）

- RLHF 需要训练独立的奖励模型，而该模型通常是规模大、成本高的 LLM
- RLVR 使用可自动验证、确定性的奖励来替代学习得到的奖励模型


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F06_raschka.webp" width=600>

- RLVR 的流行很大程度上源于 DeepSeek-R1 在 2025 年取得的成功：它无需依赖人类偏好数据或学习得到的奖励模型，也表现出了强大的推理能力
- DeepSeek-R1 使用可自动验证的奖励来训练推理行为，例如数学题的正确性检查，以及编程任务中的代码编译或执行
- 本书侧重基于数学的验证，但其核心思想与代码验证相同：使用二元成功信号自动计算奖励


&nbsp;
## 6.2 使用 GRPO 逐步理解可验证奖励强化学习

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F07_raschka.webp" width=600>

- 在介绍整体图景并了解强化学习如何融入 LLM 开发生命周期之后，下面将实现 RLVR 来训练推理模型（类似 DeepSeek-R1-Zero，但规模小得多；同等规模的运行仅 GPU 成本就要数十万美元）
- 面向 LLM 的强化学习使用所谓的策略梯度算法更新待训练的 LLM（在强化学习语境中称为“策略”）
- RLHF 中常用的策略梯度算法是近端策略优化（PPO），RLVR 也可以使用同一算法
- 不过，DeepSeek 团队训练 DeepSeek-R1 推理模型时使用了更简单的分组相对策略优化（GRPO）算法，该算法最早用于 DeepSeekMath
- GRPO 对资源更友好：PPO 需要另一个 LLM 计算价值函数，而 GRPO 不需要，因为它从一组采样回复的相对比较中获得学习信号
- 感兴趣的读者可以阅读我的文章 [The State of Reinforcement Learning for LLM Reasoning](https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training)，其中对 PPO 与 GRPO 做了更详细的并列比较
- 本章使用 GRPO 实现 RLVR
- 下一章还会介绍 GRPO 的其他改进，以提升训练稳定性和最终模型性能


### 6.2.1 通过厨师类比直观理解 GRPO

- GRPO 乍看之下可能比较复杂，因此本节先用“厨师与烹饪”的类比介绍术语和整体图景，帮助建立直观认识。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F08_raschka.webp" width=600>

- 在大多数面向 LLM 的强化学习语境中，rollout 与 completion 可以互换使用。

### 6.2.2 GRPO 的高层流程

- 后续小节实现 GRPO 的技术路线如下：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F09_raschka.webp" width=600>

&nbsp;
## 6.3 加载预训练模型

- 本章代码与前几章相同。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F10_raschka.webp" width=600>

In [ ]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

&nbsp;
## 6.4 加载 MATH 训练子集

- 我们使用从原始 MATH 数据集派生、互不重叠的训练子集；它明确排除了前几章用于模型评估的 MATH-500 样本（数据集准备详情请参阅 https://github.com/rasbt/math_full_minus_math500）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F11_raschka.webp" width=600>

- 下面的 `load_math_train` 函数与第 3 章的 [load_math500_test](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py#L422) 函数类似，只是指定了不同的文件路径。

In [ ]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [ ]:
math_train = load_math_train()

print("Dataset size:", len(math_train))

In [ ]:
from pprint import pprint

pprint(math_train[4])

- 注意，这里只需要 `"answer"` 和 `"problem"` 字段
- 理论上可能会想使用 `"solution"`，但这里希望让模型自由探索解法，而不是学习某一种特定解法和风格


&nbsp;
## 6.5 采样 rollout

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F12_raschka.webp" width=600>

- Rollout 是强化学习术语，指模型生成的回复。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F13_raschka.webp" width=400>

- 这里需要使用 `@torch.no_grad`，因为我们不希望构建计算图并通过该过程反向传播；而 `@torch.inference_mode` 会执行过多限制，无法用于此处，并会导致：

> RuntimeError：推理张量不能保存用于反向传播。请勿在自动微分跟踪的计算中使用推理模式创建的张量。要解决此问题，可以克隆张量以获得普通张量并用于自动微分，或者使用 `torch.no_grad()` 代替 `torch.inference_mode()`。


In [ ]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

- 这里没有引入新内容
- 上面的代码只是之前所开发代码的精简版：它直接结合了第 2 章的 [generate_text_basic_stream_cache](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch02.py#L57) 函数，以及第 4 章的温度采样和 top-p 采样
- 由于不需要实时打印生成的词元，现在不再逐个产出词元，而是把它们收集到张量中


In [ ]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

In [ ]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

- 实际使用时，会多次调用 `sample_response` 来生成 rollout
- 为使 GRPO 演示保持简单并与图 6.13 一致，这里改为假设模型生成了以下四个回复：


In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

&nbsp;
## 6.6 计算奖励

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F14_raschka.webp" width=400>

- 奖励就是与第 3 章类似的正确性奖励
- 不过这里还包含隐式的格式奖励：只有最终答案采用 `\boxed{}` 格式时（通过 `fallback=None`），才会获得 1.0 的奖励


In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # 要求使用 \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

- 注意：DeepSeek-R1 团队曾尝试在训练模型时使用过程奖励模型，为中间解题步骤评分
- 但这些尝试没有成功；研究人员最终认为，只使用最终答案的正确性奖励进行训练、不加入中间奖励更好


&nbsp;
## 6.7 通过优势值从 rollout 构造学习信号

- GRPO 中的“GR”（group relative，分组相对）表示：GRPO 会为每个提示词生成多个答案（rollout），并将它们相互比较以构造学习信号。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F15_raschka.webp" width=400>

- 公式非常简单：

$$\text{advantages}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$


- 其中，$r_i$ 表示第 $i$ 个 rollout 的奖励，$\mu_r$ 是这一组 rollout 的平均奖励，$\sigma_r$ 是相应的标准差，$\epsilon$ 是为提高数值稳定性、避免除零错误而加入的小常数。

In [ ]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

In [ ]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

- 如果一组中的所有奖励都相同（例如全为 0 或全为 1），那么对所有第 $i$ 个 rollout 都有 $r_i - \mu_r = 0$
- 这意味着，当所有答案都正确或都错误时，模型不会更新


&nbsp;
## 6.8 使用序列对数概率为 rollout 评分

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F16_raschka.webp" width=400>

- 上一章实现了 `avg_logprob_answer` 函数，用于计算答案词元的逐词元对数概率
- 这些平均对数概率也常被称为词元级对数概率，通常用于为 LLM 答案评分
- 评分时更倾向于取平均，因为这能对长度进行归一化
- 数学表达式为 $\frac{1}{T} \sum_{t=1}^{T} \log p_W(y_t \mid y_{<t}, x)$
- 其中，$y_1, ..., y_T$ 表示长度为 $T$ 的生成回复中的词元，$y_{<t}$ 表示此前生成的所有词元，$x$ 是输入提示词，$W$ 表示模型权重参数
- 该表达式在数学上与上一章完全相同；这里只是把 $x$ 换成 $y$，以便清楚地区分生成的输出词元和输入提示词
- 作为参考，下面复制了该函数


In [ ]:
## 第 5 章

@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # 分别编码提示词和答案词元，以便稍后获得提示词长度
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # 与前面的 calc_next_token_logprobas 相同
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # 答案词元所对应位置的索引范围
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # 与之前相同，只是改用 start 和 end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # 对答案词元分数取平均值
    return torch.mean(next_token_logps).item()

In [ ]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

- 但 GRPO 使用序列级对数概率，而不是上面经过长度归一化的词元级平均值
- 词元级平均值适合评分，因为它使不同长度的输出可比较
- 在 GRPO 中，每个 rollout 的整个序列只获得一个奖励和一个优势值；为正确缩放梯度，对数概率必须反映完整序列的似然，即对词元级对数概率求和
- 如果改为取平均，就会按序列长度隐式缩放学习信号并扭曲策略更新，对较长 rollout 尤其如此


- 去掉平均操作，并把 `torch.mean(next_token_logps)` 替换为 `torch.sum(next_token_logps)`，即可将其转换为序列级对数概率
- 反过来看，也可以把平均结果乘以答案词元数，得到未平均的值


In [ ]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

- 这些序列级对数概率会随序列长度 $T$ 线性变化
- 这意味着答案越长，对数概率总是越负
- 因而，对于两个质量相同的答案，会倾向于选择成本更低的较短答案
- 求和后的对数概率会鼓励模型更早停止


- 因此，如上所述，可以在函数中把 `torch.mean` 替换为 `torch.sum`
- 不过，上一章使用 `@torch.inference_mode()` 装饰器在推理模式下运行该函数；由于这里需要 PyTorch 跟踪并计算梯度，无论如何都必须重新定义它
- 此外，第 6.5 节的 `sample_response` 已返回 `token_ids` 和 `prompt_len`，因此可以去掉 `avg_logprob_answer` 中的编码和 `full_ids` 计算，简化代码


In [ ]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # 需要获取下一词元概率的位置
    # 这些位置对应于从位置 t 预测 token_ids[t + 1]
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # 对答案词元的对数概率求和
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

- 在 `torch.sum(next_token_logps)` 中不调用 `.item()`，是为了让 PyTorch 返回张量而非 Python 浮点数；这对梯度计算很重要
- 可以看到，所得值（-16.2998）与此前用答案词元数缩放 `avg_logprob_val` 得到的值（-16.2390）几乎相同；细微差异来自浮点舍入行为


- 下面使用 `torch.gather` 重写该函数，这种写法更符合 PyTorch 习惯，也针对 GPU 做了更好的优化
- 不过，两种函数在数学上等价


In [ ]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

- 这里的趋势是：越短、越简洁的答案会获得越高（越不负）的序列级对数概率
- 唯一包含错误数值（38 而不是 83）的答案得分最低
- 总体而言，求和后的对数概率偏向简洁且正确的输出


&nbsp;
## 6.9 通过 GRPO 损失将优势值转化为策略更新

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F17_raschka.webp" width=400>

In [ ]:
logps = torch.stack(rollout_logps)
print(logps)

In [ ]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

- 需要使用 `.detach()`，因为我们希望把 `advantages` 视为固定的学习信号；这样可以确保只通过对数概率进行反向传播
- 需要负号，是因为 PyTorch 优化器默认执行最小化，而这里希望最大化由对数概率加权的优势值


- 使用数学记号，可以把策略梯度损失写成：

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$ 表示批次中的 rollout 数量
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$ 是第 $i$ 个生成回复中的词元，该回复长度为 $T_i$
- $y_{<t}^{(i)}$ 表示该回复中此前生成的所有词元
- $x^{(i)}$ 是第 $i$ 个 rollout 对应的输入提示词
- $p_W$ 表示模型策略，即由权重 $W$ 参数化的下一词元概率分布
- $A_i$ 是分配给完整第 $i$ 个 rollout 的优势值
- 内层求和计算一个 rollout 的序列级对数概率
- 外层平均计算各 rollout 由优势值加权的对数概率


&nbsp;
## 6.10 在一个 GRPO 步骤中整合全部内容

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F18_raschka.webp" width=400>

In [ ]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # 阶段 1：生成 rollout
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # 阶段 2：计算奖励
        reward = reward_rlvr(text, example["answer"])
        
        # 阶段 4：计算对数概率
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # 阶段 2：收集所有奖励
    rewards = torch.tensor(roll_rewards, device=device)

    # 阶段 3：计算优势值
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # 阶段 4：收集所有对数概率
    logps = torch.stack(roll_logps)

    # 阶段 5：计算策略梯度损失
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # 下一章将在这里加入 KL 项

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

- 代码注释中的各阶段与 GRPO 图中的阶段相对应
- 注意，阶段 1 之后，代码注释直接采用阶段 2 和 4（而不是 3 和 4）；这样实现更简单，无需编写多个 for 循环


In [ ]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

&nbsp;
## 6.11 实现 GRPO 训练循环

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F19_raschka.webp" width=600>

- 由于资源需求本就很高，这里不使用批处理。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F20_raschka.webp" width=400>

In [ ]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # 阶段 1：初始化优化器
    # （模型已在函数外部初始化）
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # 阶段 2：迭代各训练步骤
        for step in range(steps):

            # 阶段 3：重置损失梯度
            # （最佳实践是在每个步骤开始时执行此操作）
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # 阶段 4：计算 GRPO 损失
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # 阶段 5：通过反向传播计算损失梯度
            stats["loss_tensor"].backward()

            # 裁剪过大的梯度以提升训练稳定性
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # 阶段 6：使用损失梯度更新模型权重
            optimizer.step()

            # 阶段 7：收集奖励、回复长度和损失
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # 打印每一步的指标
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # 每 10 步采样一次输出，以检查模型是否
            # 能够生成连贯文本
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # 阶段 8：保存模型检查点
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # 如果提前中断训练，则保存模型检查点
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

- 除阶段 4 的 GRPO 损失计算外，其余步骤都是训练深度神经网络（包括 LLM）时标准训练循环的一部分
- `append_csv_metrics` 会把结果记录到 CSV 文件中，以便留档并在第 7 章中进行可视化
- 如需了解使用 PyTorch 训练神经网络的一般介绍，请参阅我的文章 [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) 的第 3–8 节


In [ ]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

- 如果运行上述代码时遇到内存问题，可以减少 rollout 数量（例如 `num_rollouts=2`）以及每个 rollout 的词元数（例如 `max_new_tokens=128`）
- 但要获得相对较好的模型，至少需要 `num_rollouts=8` 和 `max_new_tokens=512`
- 如果现有硬件无法运行也不用担心，下一节将介绍如何下载预训练检查点


- 无论如何，这段代码都可能运行得很慢，因为 GRPO 是资源密集型过程
- 可以随时中断运行，程序会把最新模型检查点保存到 `checkpoints` 文件夹
- 如果想使用云端 GPU，请参阅 [GPU 云资源](../../ch02/02_setup-tips/gpu-instructions.md)文档中的建议


- 注意，这段代码不支持批量训练
- 这是有意为之：一方面可以让代码更简单、更易读，另一方面采样多个（可能很长的）rollout 本身就非常消耗资源
- 如果有多块 GPU，可以使用附加材料 [../02_rlvr_grpo_scripts_intro](../02_rlvr_grpo_scripts_intro) 中支持批处理和多 GPU 的可选版本，以更快地训练模型


&nbsp;
## 6.12 加载并评估已保存的模型检查点

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F21_raschka.webp" width=600>

- 可以使用第 2 章介绍的 `model.load_state_dict(torch.load(model_path))` 加载已保存的检查点，其中 `model_path` 指向检查点 `.pth` 文件
- 这些检查点文件也兼容第 3 章的模型评估工具
- 为方便起见，可以使用第 3 章附加材料中提供的评估脚本：

```python
uv run ../../ch03/02_math500-verifier-scripts/evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--checkpoint_path checkpoints/qwen3-0.6B-rlvr-grpo-step00050.pth
```
    

- 如果不想在自己的计算机上运行耗时较长的 GRPO 训练，也可以下载我上传到 [rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl) 的检查点（点击想下载的检查点文件，再点击[下载](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/resolve/main/grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth?download=true)按钮）
- 也可以直接在此处使用 Python 下载检查点


In [ ]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

|      | 方法                                  | 步数 | 最大词元数 | Rollout 数 | MATH-500 准确率 | 平均词元数 |
| ---- | ------------------------------------- | ---- | ---------- | ---------- | ---------------- | ---------- |
| 1    | 基础模型（第 3 章）                   | -    |            |            | 15.2%            | 78.85      |
| 2    | 推理模型（第 3 章）                   | -    |            |            | 48.2%            | 1369.79    |
| 3    | 原始 GRPO，但不含 KL（本章）          | 50   | 512        | 8          | 47.4%            | 586.11     |


- 从上表可以看到，仅训练 50 步后，由基础模型（第 1 行）初始化的已训练模型（第 3 行）就几乎与原始推理变体（第 2 行）一样好
- 需要注意，训练更长时间未必能改进模型，甚至可能使其变差，因为 GRPO 相对不稳定；下一章将介绍更多改进 GRPO 算法的技巧


&nbsp;
## 6.13 小结

- 本节没有代码